# Iterators

Every type in this chapter has been an ADT you interact with through explicit operations: `Push`, `Pop`, `Get`, `Set`. `foreach` is different — it is a loop that lets a *type* decide what "give me the next element" means, without the caller ever touching an index or a key. This section covers how that contract works, and how to implement it for your own types.

```{contents}
:local:
:depth: 2
```

```{index} IEnumerable; IEnumerator
```

## Why foreach Works

`foreach` is not built-in magic tied to `List<T>` or `Dictionary<TKey, TValue>` specifically. It works on *any* type that implements `IEnumerable<T>` — the interface that promises "I can hand out an object that walks through my elements one at a time." That helper object implements `IEnumerator<T>`, with a `MoveNext()` method and a `Current` property.

```csharp
interface IEnumerable<T>
{
    IEnumerator<T> GetEnumerator();
}

interface IEnumerator<T>
{
    T Current { get; }
    bool MoveNext();
}
```

Written out, `foreach (var item in collection)` is shorthand for a loop built from exactly these two members:

```csharp
var e = collection.GetEnumerator();
while (e.MoveNext())
{
    var item = e.Current;
    // loop body
}
```

This is the ADT pattern from earlier sections applied to iteration itself: `IEnumerable<T>` is a contract, and any representation — an array, a linked structure, a database cursor — can satisfy it.

```{index} yield return
```

## Writing an Iterator with yield return

Implementing `IEnumerator<T>` by hand — tracking a position field, writing `MoveNext()` and `Current` yourself — is tedious for something as simple as "count from one number to another." The `yield return` statement lets the compiler generate that machinery for you. A method that contains `yield return` becomes an iterator: each call to `yield return` produces one element and pauses; the method resumes from that point the next time an element is requested.

In [ ]:
static IEnumerable<int> EvenNumbers(int first, int last)
{
    for (int n = first; n <= last; n++)
    {
        if (n % 2 == 0)
        {
            yield return n;
        }
    }
}

foreach (int n in EvenNumbers(1, 10))
{
    Console.Write(n + " ");
}

Nothing about `EvenNumbers` mentions `IEnumerator`, `MoveNext`, or `Current` — the compiler builds a hidden class that implements all of it. The method's return type is `IEnumerable<int>`, and the body describes *what to produce*, not *how to track position*.

```{index} IEnumerable; custom implementation
```

## Making Your Own Type foreach-able

A standalone iterator *method* is useful, but sometimes the thing you want to iterate is a type you have already defined — an ADT you built, not a built-in collection. To make `foreach` work directly on an instance of your type, implement `IEnumerable<T>` and put a `yield return` inside `GetEnumerator()`:

In [ ]:
class NumberRange : IEnumerable<int>
{
    private readonly int start;
    private readonly int count;

    public NumberRange(int start, int count)
    {
        this.start = start;
        this.count = count;
    }

    public IEnumerator<int> GetEnumerator()
    {
        for (int i = 0; i < count; i++)
        {
            yield return start + i;
        }
    }

    IEnumerator IEnumerable.GetEnumerator() => GetEnumerator();
}

var range = new NumberRange(5, 4);
foreach (int n in range)
{
    Console.Write(n + " ");
}

`IEnumerable<T>` requires the generic `GetEnumerator()`. The second, non-generic `IEnumerator IEnumerable.GetEnumerator()` line is boilerplate C# requires because `IEnumerable<T>` itself extends the older, non-generic `IEnumerable` — it just forwards to the generic version above it. Once both are present, `NumberRange` works with `foreach`, LINQ, and any other API that accepts `IEnumerable<T>`, exactly like `List<T>` does.

```{index} iterator; lazy evaluation
```

## Iterators Are Lazy

An iterator method does not run its body when you call it — it runs one step at a time, only as far as the caller actually asks for elements. Calling `EvenNumbers(1, 1000000)` returns instantly; the loop body only executes as `foreach` requests each next value. This matters for two reasons: an iterator can describe an infinite or unbounded sequence without ever running forever, and a `foreach` that stops early (with `break`) never computes the elements it never asked for.

## Practice

1. Write an iterator method `static IEnumerable<int> CountDown(int from)` that yields `from, from - 1, ..., 1`.
2. Write a class `Pair<T>` holding two values of type `T`, implementing `IEnumerable<T>` so that `foreach` over a `Pair<T>` yields both values in order.
3. Explain, in terms of the `IEnumerable<T>` / `IEnumerator<T>` contract from earlier in this section, why a type can be `foreach`-able without being indexable — that is, without supporting `range[i]`.

```{rubric} Footnotes
```